# 05 - 端到端系统集成

本教程展示如何将所有组件整合成完整的系统。

## 学习目标
- 理解端到端流水线架构
- 掌握组件集成方法
- 学会构建完整应用

## 1. 环境准备

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
from multimodal_retriever import MultimodalRetriever, MultimodalDocument
from vision_qa_agent import VisionQAAgent
from pipeline import MultimodalRAGPipeline, PipelineConfig
from code_retriever import CodeRetriever, CodeDocument, CodeLanguage
from code_agent import CodeAgent
from review_agent import ReviewAgent

## 2. 多模态RAG流水线

In [ ]:
# 创建知识库
retriever = MultimodalRetriever()

# 添加文档
knowledge_base = [
    "Python是一种解释型、面向对象的高级编程语言",
    "机器学习是人工智能的一个分支，通过数据学习模式",
    "深度学习使用多层神经网络进行特征学习",
    "自然语言处理研究计算机与人类语言的交互",
    "计算机视觉让机器能够理解和处理图像",
]

for text in knowledge_base:
    retriever.add_document(MultimodalDocument(content=text))

print(f"知识库大小: {retriever.num_documents}")

In [ ]:
# 创建流水线
config = PipelineConfig(
    retriever_top_k=3,
    use_agent=True,
    max_agent_steps=5,
)

pipeline = MultimodalRAGPipeline(retriever=retriever, config=config)
print(f"流水线配置: {config}")

In [ ]:
# 查询
result = pipeline.query("什么是深度学习?")

print(f"问题: 什么是深度学习?")
print(f"答案: {result.answer}")
print(f"检索文档数: {len(result.retrieved_docs)}")
print(f"推理步数: {result.num_steps}")

## 3. 代码助手集成

In [ ]:
# 创建代码知识库
code_retriever = CodeRetriever()

# 添加示例代码
sample_codes = [
    ("def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[0]\n    left = [x for x in arr[1:] if x < pivot]\n    right = [x for x in arr[1:] if x >= pivot]\n    return quicksort(left) + [pivot] + quicksort(right)", "quicksort"),
    ("def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1", "binary_search"),
]

for code, name in sample_codes:
    code_retriever.add_document(CodeDocument(content=code, name=name, language=CodeLanguage.PYTHON))

print(f"代码库大小: {code_retriever.num_documents}")

In [ ]:
# 创建代码Agent和审查器
code_agent = CodeAgent(retriever=code_retriever)
reviewer = ReviewAgent()

# 生成代码
result = code_agent.generate("实现冒泡排序", language=CodeLanguage.PYTHON)
print("生成的代码:")
print(result.code)

In [ ]:
# 审查生成的代码
review_result = reviewer.review(result.code, CodeLanguage.PYTHON)
print(f"\n审查分数: {review_result.score}/100")
print(f"问题数: {len(review_result.issues)}")

## 4. 完整工作流示例

In [ ]:
class IntegratedAssistant:
    """集成助手，支持问答和代码功能。"""
    
    def __init__(self):
        # 知识检索
        self.knowledge_retriever = MultimodalRetriever()
        # 代码检索
        self.code_retriever = CodeRetriever()
        # 问答Agent
        self.qa_agent = VisionQAAgent(retriever=self.knowledge_retriever)
        # 代码Agent
        self.code_agent = CodeAgent(retriever=self.code_retriever)
        # 代码审查
        self.reviewer = ReviewAgent()
    
    def add_knowledge(self, text):
        self.knowledge_retriever.add_document(MultimodalDocument(content=text))
    
    def add_code(self, code, name):
        self.code_retriever.add_document(CodeDocument(
            content=code, name=name, language=CodeLanguage.PYTHON
        ))
    
    def ask(self, question):
        return self.qa_agent.answer(question)
    
    def generate_code(self, task):
        result = self.code_agent.generate(task, CodeLanguage.PYTHON)
        review = self.reviewer.review(result.code, CodeLanguage.PYTHON)
        return {"code": result.code, "score": review.score, "issues": review.issues}

In [ ]:
# 使用集成助手
assistant = IntegratedAssistant()

# 添加知识
assistant.add_knowledge("快速排序的平均时间复杂度是O(n log n)")
assistant.add_knowledge("冒泡排序的时间复杂度是O(n^2)")

# 问答
result = assistant.ask("哪种排序算法更快?")
print(f"问答结果: {result.answer}")

In [ ]:
# 生成代码
result = assistant.generate_code("实现选择排序")
print(f"生成代码:\n{result['code']}")
print(f"\n审查分数: {result['score']}/100")

## 5. 练习

1. 扩展IntegratedAssistant，添加图像处理功能
2. 实现代码自动修复功能
3. 添加对话历史记录

In [ ]:
# 练习空间


## 总结

本教程展示了:
- MultimodalRAGPipeline: 端到端流水线
- 组件集成: 检索+Agent+审查
- IntegratedAssistant: 完整应用示例

恭喜完成所有教程!